# Vintage Vehicle Segmentation with PCA & t-SNE

Welcome to the project on PCA and t-SNE. In this project, we will be using the auto-mpg dataset.

## Context

The shifting market conditions, globalization, cost pressure, and volatility are leading to a change in the automobile market landscape. The emergence of data, in conjunction with machine learning in automobile companies, has paved a way that is helping bring operational and business transformations.

The automobile market is vast and diverse, with numerous vehicle categories being manufactured and sold with varying configurations of attributes such as displacement, horsepower, and acceleration. We aim to find combinations of these features that can clearly distinguish certain groups of automobiles from others through this analysis, as this will inform other downstream processes for any organization aiming to sell each group of vehicles to a slightly different target audience.

You are a Data Scientist at SecondLife which is a leading used car dealership with numerous outlets across the US. Recently, they have started shifting their focus to vintage cars and have been diligently collecting data about all the vintage cars they have sold over the years. The Director of Operations at SecondLife wants to leverage the data to extract insights about the cars and find different groups of vintage cars to target the audience more efficiently.

## Objective

The objective of this problem is to explore the data, reduce the number of features by using dimensionality reduction techniques like PCA and t-SNE, and extract meaningful insights.

## Dataset

There are 8 variables in the data:

- mpg: miles per gallon

- cyl: number of cylinders

- disp: engine displacement (cu. inches) or engine size

- hp: horsepower

- wt: vehicle weight (lbs.)

- acc: time taken to accelerate from 0 to 60 mph (sec.)

- yr: model year

- car name: car model name

## Setup and Data Overview

In [ ]:
# Core data tools
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from IPython.display import display
import pandas as pd

# Visualization tools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Modeling and dimensionality reduction tools
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

RANDOM_STATE = 42

### Loading the data

In [ ]:
# Load the Auto MPG dataset
file_path = '/mnt/data/auto-mpg.csv'
df = pd.read_csv(file_path)

# Keep an untouched copy for reference
original_df = df.copy()

df.head()

### Data Overview

- Observations

- Sanity checks

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nData types and non-null counts:')
df.info()

print('\nMissing values by column:')
display(df.isna().sum().to_frame('missing_count'))

print('\nDuplicate rows:', df.duplicated().sum())
print('\nSample records:')
display(df.sample(5, random_state=RANDOM_STATE))

Dataset shape: (398, 8)

Column names:
['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'car name']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    object 
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model year    398 non-null    int64  
 7   car name      398 non-null    object 
dtypes: float64(3), int64(3), object(2)
memory usage: 25.0+ KB

Missing values by column:



Duplicate rows: 0

Sample records:


## Data Preprocessing and Exploratory Data Analysis

- EDA is an important part of any project involving data.

- It is important to investigate and understand the data better before building a model with it.

- A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.

- Missing value treatment

- Feature engineering (if needed)

- Check the correlation among the variables

- Outlier detection and treatment (if needed)

- Preparing data for modeling

- Any other preprocessing steps (if needed)

In [ ]:
# Clean horsepower, which is stored as object because missing values are encoded as '?'
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

missing_hp_before = df['horsepower'].isna().sum()
hp_median = df['horsepower'].median()
df['horsepower'] = df['horsepower'].fillna(hp_median)

# Feature engineering for business interpretation
df['model_year_full'] = 1900 + df['model year']
df['brand'] = df['car name'].str.split().str[0].str.title()

def era_bucket(year):
    if year <= 73:
        return '1970-1973'
    elif year <= 77:
        return '1974-1977'
    else:
        return '1978-1982'

df['model_year_era'] = df['model year'].apply(era_bucket)

print(f'Missing horsepower values before imputation: {missing_hp_before}')
print(f'Horsepower median used for imputation: {hp_median}')
print('Missing values after treatment:')
display(df.isna().sum().to_frame('missing_count'))

# Distribution of major categorical-like variables
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['cylinders'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Vehicle Count by Number of Cylinders')
axes[0].set_xlabel('Cylinders')
axes[0].set_ylabel('Count')

era_order = ['1970-1973', '1974-1977', '1978-1982']
df['model_year_era'].value_counts().reindex(era_order).plot(kind='bar', ax=axes[1])
axes[1].set_title('Vehicle Count by Model-Year Era')
axes[1].set_xlabel('Model-Year Era')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

brand_counts = df['brand'].value_counts().head(10).sort_values()
brand_counts.plot(kind='barh', ax=axes[2])
axes[2].set_title('Top 10 Brands by Vehicle Count')
axes[2].set_xlabel('Count')
axes[2].set_ylabel('Brand')

plt.tight_layout()
plt.show()

# Relationship between efficiency and size/power variables
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for cyl in sorted(df['cylinders'].unique()):
    sub = df[df['cylinders'] == cyl]
    axes[0].scatter(sub['weight'], sub['mpg'], label=str(cyl), alpha=0.7)
    axes[1].scatter(sub['horsepower'], sub['mpg'], label=str(cyl), alpha=0.7)
    axes[2].scatter(sub['displacement'], sub['mpg'], label=str(cyl), alpha=0.7)
axes[0].set_title('MPG vs. Weight')
axes[0].set_xlabel('Weight')
axes[0].set_ylabel('MPG')
axes[1].set_title('MPG vs. Horsepower')
axes[1].set_xlabel('Horsepower')
axes[1].set_ylabel('MPG')
axes[2].set_title('MPG vs. Displacement')
axes[2].set_xlabel('Displacement')
axes[2].set_ylabel('MPG')
axes[2].legend(title='Cylinders', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Correlation heatmap
numeric_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year']
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax)
ax.set_title('Correlation Heatmap of Vehicle Attributes')
plt.tight_layout()
plt.show()

Missing horsepower values before imputation: 6
Horsepower median used for imputation: 93.5
Missing values after treatment:


### Summary Statistics

In [ ]:
# Summary statistics for the cleaned dataset
summary_stats = df[numeric_cols].describe().T
summary_stats['range'] = summary_stats['max'] - summary_stats['min']
display(summary_stats)

# Boxplots to check distributions and possible outliers
plt.figure(figsize=(14, 8))
scaled_for_box = pd.DataFrame(StandardScaler().fit_transform(df[numeric_cols]), columns=numeric_cols)
plt.boxplot([scaled_for_box[col] for col in numeric_cols], vert=False, labels=numeric_cols)
plt.title('Standardized Boxplots for Numeric Variables')
plt.xlabel('Standardized Value')
plt.tight_layout()
plt.show()

Observations:

- The dataset contains 398 vintage vehicles and 8 original variables. Horsepower needed cleaning because six values were stored as ?; these were converted to missing values and filled with the median horsepower.

- Fuel efficiency varies widely, from very low MPG vehicles to efficient compact vintage vehicles. The average MPG is about 23.5.

- MPG is strongly negatively related to vehicle size and power variables. In particular, higher weight, displacement, horsepower, and cylinder count are associated with lower fuel economy.

- Weight, displacement, horsepower, and cylinders are strongly positively related to one another, meaning they describe a common size/power dimension in the data.

- Model year has a positive relationship with MPG, suggesting that later model-year vintage vehicles in this dataset tend to be more fuel efficient than earlier ones.

### Scaling the data

In [ ]:
# Select numeric features for PCA, t-SNE, and clustering.
# The car name and engineered brand/era variables are useful for interpretation, but not for scaling/PCA.
features = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=df.index)

X_scaled_df.head()

## Principal Component Analysis

In [ ]:
# Fit PCA using all selected numeric features
pca = PCA(random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled_df)

pca_cols = [f'PC{i+1}' for i in range(X_pca.shape[1])]
pca_df = pd.DataFrame(X_pca, columns=pca_cols, index=df.index)

explained_variance = pd.DataFrame({
    'component': pca_cols,
    'explained_variance_ratio': pca.explained_variance_ratio_,
    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_)
})

display(explained_variance)

# Scree plot
plt.figure(figsize=(10, 6))
plt.bar(explained_variance['component'], explained_variance['explained_variance_ratio'])
plt.plot(explained_variance['component'], explained_variance['cumulative_variance'], marker='o')
plt.title('PCA Explained Variance and Cumulative Variance')
plt.ylabel('Variance Share')
plt.xlabel('Principal Component')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Determine a practical number of components to retain at least 80% variance
n_components_80 = int(np.argmax(explained_variance['cumulative_variance'].values >= 0.80) + 1)
print(f'Number of principal components needed to explain at least 80% variance: {n_components_80}')

Number of principal components needed to explain at least 80% variance: 2


Observations:

- The first principal component explains the largest share of the dataset structure. In this project, PC1 explains about 71.5% of the standardized variance.

- The first two principal components explain about 83.9% of the variance, so a two-dimensional PCA map preserves most of the important information.

- This supports using PCA for visualization and clustering because the vehicle records can be summarized mainly by a smaller number of dimensions.

- From a business standpoint, PC1 mainly represents a tradeoff between fuel efficiency and vehicle size/power.

### Interpreting the Principal Components from the below DataFrame

In [ ]:
# PCA loadings show how each original variable contributes to each principal component
loadings = pd.DataFrame(
    pca.components_.T,
    columns=pca_cols,
    index=features
)

display(loadings[['PC1', 'PC2', 'PC3']])

# Visualize first three PC coefficients
loadings[['PC1', 'PC2', 'PC3']].plot(kind='bar', figsize=(14, 6))
plt.title('Feature Coefficients / Loadings for First Three Principal Components')
plt.ylabel('Coefficient Value')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Observations:

- PC1 separates efficient cars from large/powerful cars. MPG and model year load in the opposite direction from cylinders, displacement, horsepower, and weight. This means PC1 is the main efficiency vs. size/power axis.

- PC2 is influenced strongly by acceleration and model year. It helps separate vehicles that may have similar size/power profiles but differ in acceleration characteristics or era.

- PC3 captures additional variation not fully explained by PC1 and PC2, especially secondary differences among MPG, acceleration, and model year.

- Because the first two PCs already explain more than 80% of variance, the first two components are sufficient for a useful visual summary of the vehicle groups.

### Visualizing the First Two Principal Components

In [ ]:
# Use silhouette score to choose the number of clusters on the PCA representation
silhouette_results = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, algorithm='lloyd')
    labels = km.fit_predict(pca_df[['PC1', 'PC2']])
    score = silhouette_score(pca_df[['PC1', 'PC2']], labels)
    silhouette_results.append({'k': k, 'silhouette_score': score})

silhouette_df = pd.DataFrame(silhouette_results)
display(silhouette_df)

best_k = int(silhouette_df.loc[silhouette_df['silhouette_score'].idxmax(), 'k'])
print(f'Best number of clusters based on silhouette score: {best_k}')

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10, algorithm='lloyd')
df['cluster'] = kmeans.fit_predict(pca_df[['PC1', 'PC2']])
pca_df['cluster'] = df['cluster']

# Label the higher-MPG cluster as 0 and the lower-MPG/high-power cluster as 1.
cluster_mpg = df.groupby('cluster')['mpg'].mean().sort_values(ascending=False)
mapping = {cluster_mpg.index[0]: 0, cluster_mpg.index[1]: 1}
df['cluster'] = df['cluster'].map(mapping)
pca_df['cluster'] = df['cluster']

plt.figure(figsize=(10, 7))
markers = {3:'o', 4:'x', 5:'s', 6:'P', 8:'D'}
for cluster in sorted(df['cluster'].unique()):
    for cyl in sorted(df['cylinders'].unique()):
        mask = (df['cluster'] == cluster) & (df['cylinders'] == cyl)
        if mask.sum() > 0:
            plt.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                        marker=markers.get(cyl, 'o'), alpha=0.75,
                        label=f'Cluster {cluster}, {cyl} cyl')
plt.title('PCA Map of Vintage Vehicle Segments')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Cluster profiles
cluster_profile = df.groupby('cluster').agg(
    count=('mpg', 'size'),
    avg_mpg=('mpg', 'mean'),
    avg_hp=('horsepower', 'mean'),
    avg_weight=('weight', 'mean'),
    avg_displacement=('displacement', 'mean'),
    pct_8_cyl=('cylinders', lambda x: (x.eq(8).mean() * 100)),
    pct_high_mpg=('mpg', lambda x: (x.ge(30).mean() * 100)),
    pct_high_hp=('horsepower', lambda x: (x.ge(140).mean() * 100))
).round(2)

display(cluster_profile)

Best number of clusters based on silhouette score: 2


Observations:

- The strongest silhouette score supports a 2-cluster solution.

- The PCA plot shows a clear separation between the two groups along PC1, which confirms that the main split is between efficient/lighter vehicles and larger/high-power vehicles.

- Cluster 0 is the larger practical vintage segment. It has higher MPG, lower average horsepower, lower weight, and much lower displacement.

- Cluster 1 is the smaller performance/collector segment. It has lower MPG, much higher horsepower, greater weight, higher displacement, and is dominated by 8-cylinder vehicles.

- This matches the business interpretation from the supporting report: Segment 0 is best positioned for efficiency-oriented vintage buyers, while Segment 1 is best positioned for muscle-car, performance, and collector-oriented buyers.

## t-SNE

In [ ]:
# t-SNE visualization of local similarities
# t-SNE is used for visualization, not direct feature interpretation.
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=RANDOM_STATE
)
X_tsne = tsne.fit_transform(X_scaled_df)

tsne_df = pd.DataFrame(X_tsne, columns=['TSNE1', 'TSNE2'], index=df.index)
tsne_df['cluster'] = df['cluster']
tsne_df['cylinders'] = df['cylinders']

plt.figure(figsize=(10, 7))
for cluster in sorted(tsne_df['cluster'].unique()):
    sub = tsne_df[tsne_df['cluster'] == cluster]
    plt.scatter(sub['TSNE1'], sub['TSNE2'], alpha=0.75, label=f'Cluster {cluster}')
plt.title('t-SNE Map of Local Vehicle Similarities')
plt.xlabel('TSNE1')
plt.ylabel('TSNE2')
plt.legend(title='Segment')
plt.tight_layout()
plt.show()

tsne_df.head()

Observations:

- The t-SNE visualization supports the PCA-based segmentation by showing locally similar vehicles near one another.

- The high-power cluster is separated from most of the practical/efficient vehicles, which suggests the 2-cluster solution is visually meaningful.

- t-SNE does not provide direct coefficients like PCA, so it should be used as a visual validation tool rather than as the primary source for business interpretation.

- The t-SNE map confirms that vehicles with similar combinations of MPG, horsepower, weight, cylinders, displacement, acceleration, and model year form recognizable groups.

### Cluster Visualization Across Key Variables

In [ ]:
# Scatter plots by cluster
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for cluster in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == cluster]
    axes[0].scatter(sub['weight'], sub['mpg'], alpha=0.7, label=f'Cluster {cluster}')
    axes[1].scatter(sub['displacement'], sub['horsepower'], alpha=0.7, label=f'Cluster {cluster}')
    axes[2].scatter(sub['model_year_full'], sub['mpg'], alpha=0.7, label=f'Cluster {cluster}')
axes[0].set_title('Fuel Efficiency vs. Weight by Segment')
axes[0].set_xlabel('Weight')
axes[0].set_ylabel('MPG')
axes[1].set_title('Horsepower vs. Displacement by Segment')
axes[1].set_xlabel('Displacement')
axes[1].set_ylabel('Horsepower')
axes[2].set_title('MPG by Model Year and Segment')
axes[2].set_xlabel('Model Year')
axes[2].set_ylabel('MPG')
axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Boxplots for important variables
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, col in zip(axes, ['mpg', 'horsepower', 'weight', 'displacement']):
    data_to_plot = [df[df['cluster'] == c][col] for c in sorted(df['cluster'].unique())]
    ax.boxplot(data_to_plot, labels=[f'Cluster {c}' for c in sorted(df['cluster'].unique())])
    ax.set_title(f'{col.title()} by Segment')
plt.tight_layout()
plt.show()

# Era mix by segment
era_mix = pd.crosstab(df['model_year_era'], df['cluster'], normalize='index').mul(100).round(1)
display(era_mix)

era_mix = era_mix.reindex(['1970-1973','1974-1977','1978-1982'])
era_mix.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Segment Mix Shift by Model-Year Era')
plt.ylabel('Share of Era (%)')
plt.xlabel('Model-Year Era')
plt.xticks(rotation=30)
plt.legend(title='Segment')
plt.tight_layout()
plt.show()

# Top brands by segment composition
brand_segment_counts = (
    df[df['brand'].isin(df['brand'].value_counts().head(10).index)]
    .groupby(['brand', 'cluster']).size().unstack(fill_value=0)
)
brand_segment_counts['total'] = brand_segment_counts.sum(axis=1)
brand_segment_counts = brand_segment_counts.sort_values('total').drop(columns='total')

display(brand_segment_counts)
brand_segment_counts.plot(kind='barh', stacked=True, figsize=(10, 7))
plt.title('Top Brands by Segment Composition')
plt.xlabel('Vehicle Count')
plt.ylabel('Brand')
plt.legend(title='Segment')
plt.tight_layout()
plt.show()

Observations:

- The scatter plots and boxplots confirm that Cluster 0 vehicles are lighter, more fuel efficient, and generally less powerful.

- Cluster 1 vehicles are heavier, have larger displacement, have higher horsepower, and usually lower MPG.

- The model-year era comparison shows that early 1970s vehicles contain a much higher share of the high-power segment, while late 1970s and early 1980s vehicles are dominated by the efficiency-oriented segment.

- Brand composition also supports differentiated merchandising. Brands such as Toyota, Datsun, and Volkswagen are concentrated in the efficiency-oriented group, while Chevrolet, Ford, Plymouth, Dodge, AMC, Buick, and Pontiac include more mixed or performance-oriented inventory.

## Actionable Insights and Recommendations

Insights and Recommendations:

- Use a two-segment business strategy.
The PCA, t-SNE, and KMeans results support two meaningful groups of vintage vehicles: an efficiency-oriented segment and a high-power collector/performance segment.

- Position Cluster 0 as the practical vintage segment.
Cluster 0 contains the majority of the vehicles and has higher MPG, lower horsepower, lower weight, and lower displacement. SecondLife should market these vehicles to first-time classic buyers, weekend drivers, nostalgia shoppers, urban buyers, and customers who want easier ownership and better fuel economy.

- Position Cluster 1 as the performance and collector segment.
Cluster 1 is smaller but commercially important. These vehicles have lower MPG, higher horsepower, greater displacement, heavier weight, and a very high share of 8-cylinder cars. These cars should be marketed using language around V8 heritage, muscle identity, engine character, nostalgia, collectability, and period-correct performance.

- Use model year as a quick merchandising cue.
Earlier 1970–1973 vehicles are more likely to support performance narratives, while later 1978–1982 vehicles are more likely to support efficient classic ownership narratives.

- Create two listing templates.

The efficiency template should highlight MPG, lower weight, ease of ownership, affordability, and daily usability.
The power template should highlight engine size, cylinder count, horsepower, acceleration feel, collector appeal, and the emotional story of the car.

- Apply brand-specific messaging.
Toyota, Datsun, and Volkswagen should generally be promoted as practical and approachable vintage vehicles. Brands that span both clusters, such as Chevrolet, Ford, Plymouth, Dodge, AMC, Buick, and Pontiac, should be marketed according to the individual vehicle's attributes rather than brand alone.

- Improve future data collection.
For stronger pricing and profitability decisions, SecondLife should collect purchase price, sale price, gross margin, listing days, vehicle condition, mileage, restoration status, region, and customer lead source.

Final takeaway: SecondLife should not treat all vintage vehicles as one market. Efficient vintage cars can build sales volume and accessibility, while high-power classics can build excitement, brand image, and premium positioning.